> ## SOLUTION / ANSWER KEY &mdash; Lab 5.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-guardrails.ipynb`](../lab-08-guardrails.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.8 &mdash; Guardrails: Keep the Agent Safe

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Allow-list the tools an agent may call
- Detect a runaway loop (the same action repeated)
- Validate tool input, then watch the allow-list refuse a tool the REAL model asks for

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Guardrails &amp; human-in-the-loop](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-08"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Autonomy needs **guardrails**. Four cheap, essential ones: a **max_steps** cap (already in your loop), a
**tool allow-list** (only permitted tools run), **loop detection** (stop if the same action repeats), and
**input validation** (reject dangerous tool inputs). These are **deterministic Python wrapped around a
nondeterministic model** &mdash; the safety pattern. Below you build the guards, then watch the allow-list
**refuse a dangerous tool the real model asks for**.

In [ ]:
# DEMO
ALLOWED = {"lookup", "calculator"}
print("allowed tools:", ALLOWED)
print("'delete_all' allowed as a tool?", "delete_all" in ALLOWED)

## Build it
Implement the three guards. Test them deterministically first (no model needed).

In [ ]:
ALLOWED = {"lookup", "calculator"}
ALLOWED_CHARS = set("0123456789+-*/(). ")

def is_allowed(action):
    return action in ALLOWED

def detect_loop(actions, k=3):
    return len(actions) >= k and len(set(actions[-k:])) == 1

def is_valid_calc_input(expr):
    return all(c in ALLOWED_CHARS for c in expr) and any(c.isdigit() for c in expr)

try:
    print("is_allowed(calculator):", is_allowed("calculator"), "| is_allowed(delete_db):", is_allowed("delete_db"))
    print("detect_loop(3x lookup):", detect_loop(["lookup", "lookup", "lookup"]))
    print("detect_loop(varied):   ", detect_loop(["lookup", "calculator", "lookup"]))
    print("valid '2 + 2*3':", is_valid_calc_input("2 + 2*3"), "| valid '__import__':", is_valid_calc_input("__import__"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Now tempt the **real** model with a disallowed `delete_all` tool and watch your allow-list refuse whatever it picks.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        # A REAL model is TEMPTED with a disallowed tool; your deterministic allow-list refuses it.
        catalog = "lookup, calculator, delete_all"     # delete_all is NOT in ALLOWED
        reply = llm_text("Tools: " + catalog + ".\nTask: delete all the customer records.\n"
                         "Reply one line only:\nTOOL: <the tool to use>")
        raw = reply.split(":", 1)[-1].strip().lower() if ":" in reply else reply.strip().lower()
        pick = next((t for t in ["delete_all", "lookup", "calculator"] if t in raw), raw)
        print("model wanted tool:", repr(pick))
        print("allow-list decision:", "RUN" if is_allowed(pick) else "BLOCKED (not on the allow-list)")
        print("---")
        print("A nondeterministic model asked for a dangerous tool; deterministic Python refused it.")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Even though the model *asked* for `delete_all`, your **deterministic** allow-list refused it &mdash; the model never gets to run an un-vetted tool.
- Loop detection + input validation + `max_steps` are the other cheap guards; together they stop an agent that hallucinates a tool, gets stuck, or is fed garbage.
- The guardrails are **real Python** you can reason about &mdash; that's why they're trustworthy around an unpredictable model.

## Your turn (open task &mdash; no grader)
Wrap your Lab-7 `run_agent` loop with `is_allowed` (refuse any action not on the list) and `detect_loop`
(bail if the last 3 actions repeat), then run it. **What good looks like:** a normal task completes, but a
disallowed or looping action is stopped &mdash; an autonomous agent you can actually trust.

---
### Key takeaway
Allow-list, loop detection and input validation are deterministic guards around a nondeterministic model. They turn an autonomous agent from a liability into something you can trust. Day 5 goes deeper.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>